In [ ]:
#Boltuix -dataset

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from datasets import load_dataset




In [2]:

VOCAB_SIZE = 12000
EMBEDDING_DIM = 100
HIDDEN_DIM = 128
OUTPUT_DIM = 13

BATCH_SIZE = 64
EPOCHS = 5
MAX_LEN = 60


In [3]:
print("Loading boltuix/emotions-dataset...")

dataset = load_dataset("boltuix/emotions-dataset")


Loading boltuix/emotions-dataset...


In [4]:
X_train_raw = list(dataset["train"]["Sentence"])
y_train_strings = list(dataset["train"]["Label"])


In [5]:
#Label Mapping

In [6]:


label_to_id = {
    "sadness": 0,
    "happiness": 1,
    "love": 2,
    "anger": 3,
    "fear": 4,
    "surprise": 5,
    "shame": 6,
    "guilt": 7,
    "disgust": 8,
    "confusion": 9,
    "boredom": 10,
    "relief": 11,
    "sarcasm": 12
}


In [7]:
#text label to integer

In [8]:

y_all_labels = np.array([
    label_to_id.get(
        str(label).lower().strip(),
        0
    )
    for label in y_train_strings
], dtype=np.int32)

In [9]:
if "validation" in dataset:

    X_val_raw = list(dataset["validation"]["Sentence"])

    y_val_strings = list(dataset["validation"]["Label"])

    y_val = np.array([
        label_to_id.get(
            str(label).lower().strip(),
            0
        )
        for label in y_val_strings
    ], dtype=np.int32)

    y_train = y_all_labels

else:

    total_len = len(X_train_raw)

    # 10% validation data
    val_size = int(total_len * 0.1)

    split_idx = total_len - val_size

    # Split text
    X_val_raw = X_train_raw[split_idx:]
    X_train_raw = X_train_raw[:split_idx]

    # Split labels
    y_val = y_all_labels[split_idx:]
    y_train = y_all_labels[:split_idx]


print(
    f"Training samples: {len(X_train_raw)}"
)

print(
    f"Validation samples: {len(X_val_raw)}"
)




Training samples: 118176
Validation samples: 13130


In [10]:
#Tokenization

In [11]:
print("\nTokenizing text...")

VOCAB_SIZE = 12000
 
tokenizer = Tokenizer(
    num_words=VOCAB_SIZE,
    oov_token="<OOV>"
)


tokenizer.fit_on_texts(X_train_raw)


X_train_seq = tokenizer.texts_to_sequences(X_train_raw)


X_val_seq = tokenizer.texts_to_sequences(X_val_raw)

print("Tokenization completed.")
print("First training sequence:", X_train_seq[0])
print("First validation sequence:", X_val_seq[0])


Tokenizing text...
Tokenization completed.
First training sequence: [837, 497, 708, 61, 850, 9177, 1925, 65, 6319, 3400, 1, 2413, 3, 1, 2007, 3, 6597, 828, 18, 426]
First validation sequence: [73, 124, 74, 7, 2077, 43, 74, 10897, 108, 47, 2077, 153, 54]


In [12]:
#Padding

In [13]:

print("Padding sequences...")

X_train = pad_sequences(
    X_train_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_val = pad_sequences(
    X_val_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)


print("Preprocessing complete.")

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)




Padding sequences...
Preprocessing complete.
X_train shape: (118176, 60)
X_val shape: (13130, 60)


In [14]:
#Build LSTM Model 

In [15]:

print("\nBuilding LSTM model...")

model = Sequential([

        tf.keras.layers.Input(
        shape=(MAX_LEN,)
    ),

       Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),

   
    LSTM(
        units=HIDDEN_DIM,
        return_sequences=False
    ),

    Dropout(0.3),

   
    Dense(
        units=OUTPUT_DIM,
        activation="softmax"
    )
])





Building LSTM model...


In [16]:
#Compile Model

In [19]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()



Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 60, 100)        │     1,200,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │       117,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 13)             │         1,677 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,318,925 (5.03 MB)

 Trainable params: 1,318,925 (5.03 MB)

 Non-trainable params: 0 (0.00 B)

In [20]:
#Train Model

In [21]:
print("\nStarting training...")

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE
)





Starting training...
Epoch 1/5
1847/1847 ━━━━━━━━━━━━━━━━━━━━ 230s 122ms/step - accuracy: 0.2682 - loss: 2.0759 - val_accuracy: 0.2725 - val_loss: 2.0581
Epoch 2/5
1847/1847 ━━━━━━━━━━━━━━━━━━━━ 233s 126ms/step - accuracy: 0.2729 - loss: 2.0601 - val_accuracy: 0.2742 - val_loss: 1.9519
Epoch 3/5
1847/1847 ━━━━━━━━━━━━━━━━━━━━ 220s 119ms/step - accuracy: 0.4541 - loss: 1.5846 - val_accuracy: 0.5599 - val_loss: 1.2869
Epoch 4/5
1847/1847 ━━━━━━━━━━━━━━━━━━━━ 222s 120ms/step - accuracy: 0.6030 - loss: 1.1557 - val_accuracy: 0.6192 - val_loss: 1.1041
Epoch 5/5
1847/1847 ━━━━━━━━━━━━━━━━━━━━ 227s 123ms/step - accuracy: 0.6528 - loss: 1.0043 - val_accuracy: 0.6360 - val_loss: 1.0734


In [22]:
#Emotion Label Mapping

In [23]:
label_mapping = {
    0: "sadness",
    1: "happiness",
    2: "love",
    3: "anger",
    4: "fear",
    5: "surprise",
    6: "shame",
    7: "guilt",
    8: "disgust",
    9: "confusion",
    10: "boredom",
    11: "relief",
    12: "sarcasm"
}




In [24]:
#Build the prediction function

In [25]:
def predict_emotion(text):

    # Convert text to sequence
    sequence = tokenizer.texts_to_sequences([text])

    # Pad sequence
    padded_sequence = pad_sequences(
        sequence,
        maxlen=MAX_LEN,
        padding="post",
        truncating="post"
    )

    # Predict
    prediction = model.predict(
        padded_sequence,
        verbose=0
    )

    # Get class with highest probability
    predicted_class = np.argmax(prediction[0])

    # Convert class ID to emotion
    emotion = label_mapping[predicted_class]

    # Get confidence
    confidence = prediction[0][predicted_class]

    return emotion, confidence

In [26]:
#Add sentiment classification

In [27]:
def get_sentiment(emotion):

    if emotion in [
        "happiness",
        "love",
        "relief"
    ]:
        return "Positive"

    elif emotion in [
        "sadness",
        "anger",
        "fear",
        "shame",
        "guilt",
        "disgust",
        "boredom"
    ]:
        return "Negative"

    else:
        return "Neutral/Complex"

In [28]:
#combine emotion + sentiment

In [29]:
def predict_emotion_and_sentiment(text):

    emotion, confidence = predict_emotion(text)

    sentiment = get_sentiment(emotion)

    return emotion, sentiment, confidence

In [30]:
#Test your trained model

In [31]:
text = "I am extremely happy today."

emotion, sentiment, confidence = predict_emotion_and_sentiment(text)

print("\n================================")
print("       MODEL PREDICTION")
print("================================")

print("Input Text:", text)
print("Predicted Emotion:", emotion)
print("Sentiment:", sentiment)
print("Confidence:", round(float(confidence) * 100, 2), "%")


       MODEL PREDICTION
Input Text: I am extremely happy today.
Predicted Emotion: happiness
Sentiment: Positive
Confidence: 95.83 %


In [32]:
text = "I felt so much guilt and confusion after making that choice."

emotion, sentiment, confidence = predict_emotion_and_sentiment(text)

print("\nInput Text:", text)
print("Predicted Emotion:", emotion)
print("Sentiment:", sentiment)
print("Confidence:", round(float(confidence) * 100, 2), "%")


Input Text: I felt so much guilt and confusion after making that choice.
Predicted Emotion: guilt
Sentiment: Negative
Confidence: 58.42 %


In [33]:
test_sentences = [
    "I am extremely happy today.",
    "I felt so much guilt after making that choice.",
    "I am scared about what will happen.",
    "I am very angry about this situation.",
    "I love spending time with my family.",
    "I am confused about what I should do.",
    "I feel bored with everything today."
]

for text in test_sentences:

    emotion, sentiment, confidence = predict_emotion_and_sentiment(text)

    print("\n--------------------------------")
    print("Text:", text)
    print("Emotion:", emotion)
    print("Sentiment:", sentiment)
    print("Confidence:", round(float(confidence) * 100, 2), "%")


--------------------------------
Text: I am extremely happy today.
Emotion: happiness
Sentiment: Positive
Confidence: 95.83 %

--------------------------------
Text: I felt so much guilt after making that choice.
Emotion: guilt
Sentiment: Negative
Confidence: 70.89 %

--------------------------------
Text: I am scared about what will happen.
Emotion: fear
Sentiment: Negative
Confidence: 88.43 %

--------------------------------
Text: I am very angry about this situation.
Emotion: anger
Sentiment: Negative
Confidence: 85.92 %

--------------------------------
Text: I love spending time with my family.
Emotion: love
Sentiment: Positive
Confidence: 84.9 %

--------------------------------
Text: I am confused about what I should do.
Emotion: confusion
Sentiment: Neutral/Complex
Confidence: 32.85 %

--------------------------------
Text: I feel bored with everything today.
Emotion: sadness
Sentiment: Negative
Confidence: 45.61 %
